# Two-Stage Joint Hyperparameter Search

This notebook runs a **coarse joint sweep** followed by a **narrow sweep around the coarse best**.

Searched together:
- `batch_size`
- `lr_decay`
- `embed_dim`
- `cross_batch.amplification_rate`
- `lr`

Fixed winners applied to all runs:
- `activation = silu`
- `dropout = 0.01`
- `norm = null`
- `lr_category = null`
- `lr_category_decay = null` (implemented as neutral `1.0` in training because trainer expects a float)

Input config: `configs/hyper.json` (dataset + per-source behavior still come from config).


In [ ]:
%matplotlib inline
from __future__ import annotations

import copy
import itertools
import json
import math
import sys
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "hyper.json").exists() and (candidate / "localized_entropy").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing configs/hyper.json")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from localized_entropy.config import (
    get_data_source,
    get_ctr_dataset_name,
    get_training_source,
    load_and_resolve,
    loss_label,
)
from localized_entropy.experiments import build_loss_loaders, build_model, train_single_loss
from localized_entropy.hyper_search import build_search_context, collect_all_data_metrics
from localized_entropy.training import compute_base_rates_from_loader, evaluate
from localized_entropy.utils import set_seed

np.set_printoptions(precision=6, suppress=True)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 80)


In [ ]:
# ------------------------------
# Notebook flags
# ------------------------------
LOSS_MODE = "le"  # one of: bce, le, fl
CONFIG_PATH = REPO_ROOT / "configs" / "hyper.json"

# Search size/sparsity controls
COARSE_POINTS_PER_PARAM = 6
COARSE_MAX_TRIALS = 120
COARSE_USE_FULL_GRID = False
COARSE_EPOCHS = 5

NARROW_POINTS_PER_PARAM = 5
NARROW_MAX_TRIALS = 72
NARROW_USE_FULL_GRID = False
NARROW_EPOCHS = 5

SWEEP_RANDOM_SEED = 42

# Value bounds / ladders
BATCH_SIZE_LADDER = [16384, 25000, 32768, 65536]
EMBED_DIM_LADDER = [4]
AMPLIFICATION_LADDER = [1.0, 4.0, 16.0, 32.0, 64.0]
LR_DECAY_LADDER = [0.999, 0.9995, 0.9999, 1.0]

LR_MIN = 1e-5
LR_MAX = 5e-2

# Fixed winning params from prior single-parameter sweeps
FIXED_MODEL_PARAMS = {
    "activation": "silu",
    "dropout": 0.01,
    "norm": None,
}


In [ ]:
LOSS_MODE_MAP = {
    "bce": "bce",
    "le": "localized_entropy",
    "fl": "focal",
}
if LOSS_MODE not in LOSS_MODE_MAP:
    raise ValueError(f"LOSS_MODE must be one of: {sorted(LOSS_MODE_MAP)}")

LOSS_MODE_CANONICAL = LOSS_MODE_MAP[LOSS_MODE]
LOSS_LABEL = loss_label(LOSS_MODE_CANONICAL)
ctx = build_search_context(str(CONFIG_PATH), loss_mode=LOSS_MODE_CANONICAL)

resolved_cfg = load_and_resolve(str(CONFIG_PATH))
source = get_data_source(resolved_cfg)
if source == "ctr":
    dataset_key = get_ctr_dataset_name(resolved_cfg)
else:
    dataset_key = get_training_source(resolved_cfg)

loss_folder = {
    "bce": "bce",
    "localized_entropy": "le",
    "focal": "focal",
}[LOSS_MODE_CANONICAL]

output_root = REPO_ROOT / "output" / "hyper_search"
all_losses = ["bce", "le", "focal"]
all_datasets = ["synthetic", "avazu", "criteo", "yambda"]
for loss_name in all_losses:
    for ds_name in all_datasets:
        (output_root / loss_name / ds_name).mkdir(parents=True, exist_ok=True)

run_output_dir = output_root / loss_folder / dataset_key
run_output_dir.mkdir(parents=True, exist_ok=True)
run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Loss mode: {LOSS_MODE} ({LOSS_LABEL})")
print(f"Dataset key: {dataset_key}")
print(f"Eval split: {ctx.eval_name}")
print(f"Output directory: {run_output_dir}")


In [ ]:
SEARCH_PARAM_ORDER = ["batch_size", "lr_decay", "embed_dim", "amplification_rate", "lr"]
SORT_BY = [
    "ece_small",
    "per_condition_calibration_error",
    "ece",
    "global_calibration_error",
    "test_loss",
    "batch_size",
    "lr_decay",
    "embed_dim",
    "amplification_rate",
    "lr",
    "epoch",
]
ASCENDING = [True] * len(SORT_BY)


def nearest_window_from_ladder(center: float, ladder: List[float], n: int) -> List[float]:
    n = max(1, int(n))
    vals = sorted(set(float(v) for v in ladder))
    idx = min(range(len(vals)), key=lambda i: abs(vals[i] - float(center)))
    half = n // 2
    start = max(0, idx - half)
    end = min(len(vals), start + n)
    start = max(0, end - n)
    return vals[start:end]


def geometric_window(center: float, n: int, factor: float = 2.0, min_value: float = 1e-6, max_value: float = 1.0) -> List[float]:
    n = max(1, int(n))
    center = float(np.clip(center, min_value, max_value))
    if n == 1:
        return [center]
    radius = n // 2
    exponents = list(range(-radius, radius + 1))
    if len(exponents) > n:
        exponents = exponents[:n]
    vals = [float(np.clip(center * (factor ** e), min_value, max_value)) for e in exponents]
    vals = sorted(set(vals))
    while len(vals) < n:
        vals.append(vals[-1])
        vals = sorted(set(vals))
        if len(vals) == 1:
            break
    return vals[:n]


def sample_grid(space: Dict[str, List[Any]], max_trials: int | None, seed: int, use_full_grid: bool) -> List[Dict[str, Any]]:
    keys = list(space.keys())
    combos = [
        dict(zip(keys, values))
        for values in itertools.product(*(space[k] for k in keys))
    ]
    if use_full_grid or max_trials is None or len(combos) <= int(max_trials):
        return combos
    rng = np.random.default_rng(int(seed))
    idx = rng.choice(len(combos), size=int(max_trials), replace=False)
    idx = np.sort(idx)
    return [combos[int(i)] for i in idx]


def rows_best_epoch_per_config(df: pd.DataFrame, param_cols: List[str]) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    sorted_df = df.sort_values(SORT_BY, ascending=ASCENDING).reset_index(drop=True)
    return sorted_df.drop_duplicates(subset=param_cols, keep="first").reset_index(drop=True)


def base_value_from_context(ctx_obj):
    lt = ctx_obj.loss_train_cfg
    cfg = ctx_obj.cfg
    model_cfg = cfg.get("model", {})
    base_lr = float(lt.get("lr", cfg.get("training", {}).get("lr", 1e-3)))
    base_batch = int(lt.get("batch_size", cfg.get("training", {}).get("batch_size", 25000)))
    base_lr_decay = float(lt.get("lr_decay", cfg.get("training", {}).get("lr_decay", 1.0)))
    base_embed = int(model_cfg.get("embed_dim", 16))
    cb_cfg = lt.get("cross_batch") if isinstance(lt.get("cross_batch"), dict) else {}
    base_amp = float(cb_cfg.get("amplification_rate", 1.0))
    return {
        "lr": base_lr,
        "batch_size": base_batch,
        "lr_decay": base_lr_decay,
        "embed_dim": base_embed,
        "amplification_rate": base_amp,
    }


def build_coarse_space(ctx_obj) -> Dict[str, List[Any]]:
    base = base_value_from_context(ctx_obj)
    space = {
        "batch_size": [int(v) for v in nearest_window_from_ladder(base["batch_size"], BATCH_SIZE_LADDER, COARSE_POINTS_PER_PARAM)],
        "lr_decay": [float(v) for v in nearest_window_from_ladder(base["lr_decay"], LR_DECAY_LADDER, COARSE_POINTS_PER_PARAM)],
        "embed_dim": [int(v) for v in nearest_window_from_ladder(base["embed_dim"], EMBED_DIM_LADDER, COARSE_POINTS_PER_PARAM)],
        "amplification_rate": [float(v) for v in nearest_window_from_ladder(base["amplification_rate"], AMPLIFICATION_LADDER, COARSE_POINTS_PER_PARAM)],
        "lr": [float(v) for v in geometric_window(base["lr"], COARSE_POINTS_PER_PARAM, factor=2.0, min_value=LR_MIN, max_value=LR_MAX)],
    }
    if LOSS_MODE_CANONICAL != "localized_entropy":
        space["amplification_rate"] = [1.0]
    return space


def build_narrow_space(best_row: pd.Series) -> Dict[str, List[Any]]:
    best = {k: best_row[k] for k in SEARCH_PARAM_ORDER}
    space = {
        "batch_size": [int(v) for v in nearest_window_from_ladder(float(best["batch_size"]), BATCH_SIZE_LADDER, NARROW_POINTS_PER_PARAM)],
        "lr_decay": [float(v) for v in nearest_window_from_ladder(float(best["lr_decay"]), LR_DECAY_LADDER, NARROW_POINTS_PER_PARAM)],
        "embed_dim": [int(v) for v in nearest_window_from_ladder(float(best["embed_dim"]), EMBED_DIM_LADDER, NARROW_POINTS_PER_PARAM)],
        "amplification_rate": [float(v) for v in nearest_window_from_ladder(float(best["amplification_rate"]), AMPLIFICATION_LADDER, NARROW_POINTS_PER_PARAM)],
        "lr": [float(v) for v in geometric_window(float(best["lr"]), NARROW_POINTS_PER_PARAM, factor=1.6, min_value=LR_MIN, max_value=LR_MAX)],
    }
    if LOSS_MODE_CANONICAL != "localized_entropy":
        space["amplification_rate"] = [1.0]
    return space


In [ ]:
def run_joint_search(
    ctx_obj,
    param_grid: List[Dict[str, Any]],
    *,
    stage_name: str,
    stage_epochs: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    epoch_records: List[Dict[str, Any]] = []
    condition_records: List[Dict[str, Any]] = []

    for trial_id, params in enumerate(param_grid):
        cfg_run = copy.deepcopy(ctx_obj.cfg)

        cfg_run.setdefault("model", {})
        cfg_run["model"].update(FIXED_MODEL_PARAMS)
        cfg_run["model"]["embed_dim"] = int(params["embed_dim"])

        cfg_run.setdefault("training", {})
        cfg_run["training"]["batch_size"] = int(params["batch_size"])
        cfg_run["training"]["lr"] = float(params["lr"])
        cfg_run["training"]["lr_decay"] = float(params["lr_decay"])
        cfg_run["training"]["lr_category"] = None
        cfg_run["training"]["lr_category_decay"] = None

        by_loss = cfg_run.get("training", {}).get("by_loss", {})
        if isinstance(by_loss, dict):
            loss_cfg = by_loss.get(ctx_obj.loss_mode)
            if isinstance(loss_cfg, dict):
                loss_cfg["batch_size"] = int(params["batch_size"])
                loss_cfg["lr"] = float(params["lr"])
                loss_cfg["lr_decay"] = float(params["lr_decay"])
                loss_cfg["lr_category"] = None
                loss_cfg["lr_category_decay"] = None
                by_source = loss_cfg.get("by_source")
                if isinstance(by_source, dict):
                    for src_name, src_cfg in by_source.items():
                        if not isinstance(src_cfg, dict):
                            continue
                        src_cfg["batch_size"] = int(params["batch_size"])
                        src_cfg["lr"] = float(params["lr"])
                        src_cfg["lr_decay"] = float(params["lr_decay"])
                        src_cfg["lr_category"] = None
                        src_cfg["lr_category_decay"] = None
                        if ctx_obj.loss_mode == "localized_entropy":
                            cb_cfg = src_cfg.get("cross_batch") if isinstance(src_cfg.get("cross_batch"), dict) else {}
                            cb_cfg["enabled"] = True
                            cb_cfg["amplification_rate"] = float(params["amplification_rate"])
                            src_cfg["cross_batch"] = cb_cfg

        set_seed(
            int(cfg_run["project"]["seed"]),
            ctx_obj.use_cuda,
            deterministic_cuda=bool(cfg_run.get("device", {}).get("deterministic_cuda", False)),
        )

        loss_loaders, loss_train_cfg = build_loss_loaders(
            cfg_run,
            ctx_obj.loss_mode,
            ctx_obj.splits,
            ctx_obj.device,
            ctx_obj.use_cuda,
            ctx_obj.use_mps,
        )

        base_rates_trial = None
        if ctx_obj.loss_mode == "localized_entropy":
            first_param_dtype = torch.float64 if ctx_obj.model_dtype == torch.float64 else torch.float32
            base_rates_trial = compute_base_rates_from_loader(
                loss_loaders.train_loader,
                num_conditions=int(ctx_obj.splits.num_conditions),
                device=ctx_obj.device,
                dtype=first_param_dtype,
                non_blocking=ctx_obj.non_blocking,
            )

        focal_cfg = loss_train_cfg.get("focal", {}) if isinstance(loss_train_cfg, dict) else {}
        focal_alpha = float(focal_cfg.get("alpha", 0.25)) if ctx_obj.loss_mode == "focal" else None
        focal_gamma = float(focal_cfg.get("gamma", 2.0)) if ctx_obj.loss_mode == "focal" else None

        le_cross_batch_cfg = None
        if ctx_obj.loss_mode == "localized_entropy":
            cb_cfg = loss_train_cfg.get("cross_batch") if isinstance(loss_train_cfg.get("cross_batch"), dict) else {}
            cb_cfg = dict(cb_cfg)
            cb_cfg["enabled"] = True
            cb_cfg["amplification_rate"] = float(params["amplification_rate"])
            le_cross_batch_cfg = cb_cfg

        model = build_model(cfg_run, ctx_obj.splits, ctx_obj.device, dtype=ctx_obj.model_dtype)

        payload = {
            "stage": str(stage_name),
            "trial_id": int(trial_id),
            "batch_size": int(params["batch_size"]),
            "lr_decay": float(params["lr_decay"]),
            "embed_dim": int(params["embed_dim"]),
            "amplification_rate": float(params["amplification_rate"]),
            "lr": float(params["lr"]),
        }

        def on_epoch_eval(_eval_preds: np.ndarray, epoch: int, p=payload):
            eval_kwargs = {}
            if ctx_obj.loss_mode == "localized_entropy" and base_rates_trial is not None:
                eval_kwargs["base_rates"] = base_rates_trial
            if ctx_obj.loss_mode == "focal":
                eval_kwargs["focal_alpha"] = focal_alpha
                eval_kwargs["focal_gamma"] = focal_gamma

            eval_loss, _ = evaluate(
                model,
                ctx_obj.eval_loader,
                ctx_obj.device,
                loss_mode=ctx_obj.loss_mode,
                non_blocking=ctx_obj.non_blocking,
                **eval_kwargs,
            )
            metrics, per_cond_df = collect_all_data_metrics(
                cfg_run,
                model,
                eval_loader=ctx_obj.eval_loader,
                eval_labels=ctx_obj.eval_labels,
                eval_conds=ctx_obj.eval_conds,
                device=ctx_obj.device,
                non_blocking=ctx_obj.non_blocking,
            )

            row = dict(p)
            row.update(
                {
                    "epoch": int(epoch),
                    "loss_mode": LOSS_MODE,
                    "test_loss": float(eval_loss),
                    "global_calibration": float(metrics["global_calibration"]),
                    "global_calibration_error": float(metrics["global_calibration_error"]),
                    "ece": float(metrics["ece"]),
                    "ece_small": float(metrics["ece_small"]),
                    "per_condition_calibration_error": float(metrics["per_condition_calibration_error"]),
                    "per_condition_calibration_error_macro": float(metrics["per_condition_calibration_error_macro"]),
                    "small_prob_max": float(metrics["small_prob_max"]),
                }
            )
            epoch_records.append(row)

            for _, cond in per_cond_df.iterrows():
                c_row = dict(p)
                c_row.update(
                    {
                        "epoch": int(epoch),
                        "condition": int(cond["condition"]),
                        "count": int(cond["count"]),
                        "base_rate": float(cond["base_rate"]),
                        "pred_mean": float(cond["pred_mean"]),
                        "calibration": float(cond["calibration"]),
                        "calibration_abs_error": float(cond["calibration_abs_error"]),
                        "ece": float(cond["ece"]),
                        "ece_small": float(cond["ece_small"]),
                    }
                )
                condition_records.append(c_row)

        train_single_loss(
            model=model,
            loss_mode=ctx_obj.loss_mode,
            train_loader=loss_loaders.train_loader,
            train_eval_loader=ctx_obj.eval_loader,
            eval_loader=ctx_obj.eval_loader,
            device=ctx_obj.device,
            epochs=int(stage_epochs),
            lr=float(params["lr"]),
            lr_category=None,
            lr_decay=float(params["lr_decay"]),
            lr_category_decay=1.0,
            lr_zero_after_epochs=loss_train_cfg.get("lr_zero_after_epochs"),
            eval_has_labels=True,
            le_base_rates_train=base_rates_trial if ctx_obj.loss_mode == "localized_entropy" else None,
            le_base_rates_train_eval=base_rates_trial if ctx_obj.loss_mode == "localized_entropy" else None,
            le_base_rates_eval=base_rates_trial if ctx_obj.loss_mode == "localized_entropy" else None,
            focal_alpha=focal_alpha,
            focal_gamma=focal_gamma,
            non_blocking=ctx_obj.non_blocking,
            eval_callback=on_epoch_eval,
            plot_eval_hist_epochs=False,
            print_embedding_table=False,
            le_cross_batch_cfg=le_cross_batch_cfg if ctx_obj.loss_mode == "localized_entropy" else None,
        )

    results_df = pd.DataFrame(epoch_records)
    condition_df = pd.DataFrame(condition_records)
    if results_df.empty:
        raise RuntimeError(f"{stage_name}: no results were produced")
    return results_df.sort_values(SORT_BY, ascending=ASCENDING).reset_index(drop=True), condition_df


In [ ]:
coarse_space = build_coarse_space(ctx)
coarse_grid = sample_grid(
    coarse_space,
    max_trials=COARSE_MAX_TRIALS,
    seed=SWEEP_RANDOM_SEED,
    use_full_grid=COARSE_USE_FULL_GRID,
)

print("Coarse space:")
for k, v in coarse_space.items():
    print(f"- {k}: {v}")
print(f"Coarse trial count: {len(coarse_grid)}")

coarse_results_df, coarse_condition_df = run_joint_search(
    ctx,
    coarse_grid,
    stage_name="coarse",
    stage_epochs=COARSE_EPOCHS,
)
coarse_best_per_cfg = rows_best_epoch_per_config(coarse_results_df, SEARCH_PARAM_ORDER)
coarse_best_row = coarse_best_per_cfg.sort_values(SORT_BY, ascending=ASCENDING).iloc[0]

print("\nBest coarse setting:")
print(coarse_best_row[SEARCH_PARAM_ORDER + ["epoch", "ece_small", "per_condition_calibration_error", "ece", "global_calibration_error", "test_loss"]])


In [ ]:
narrow_space = build_narrow_space(coarse_best_row)
narrow_grid = sample_grid(
    narrow_space,
    max_trials=NARROW_MAX_TRIALS,
    seed=SWEEP_RANDOM_SEED + 1,
    use_full_grid=NARROW_USE_FULL_GRID,
)

print("Narrow space:")
for k, v in narrow_space.items():
    print(f"- {k}: {v}")
print(f"Narrow trial count: {len(narrow_grid)}")

narrow_results_df, narrow_condition_df = run_joint_search(
    ctx,
    narrow_grid,
    stage_name="narrow",
    stage_epochs=NARROW_EPOCHS,
)
narrow_best_per_cfg = rows_best_epoch_per_config(narrow_results_df, SEARCH_PARAM_ORDER)


In [ ]:
all_results_df = pd.concat([coarse_results_df, narrow_results_df], ignore_index=True)
all_condition_df = pd.concat([coarse_condition_df, narrow_condition_df], ignore_index=True)
all_best_per_cfg = rows_best_epoch_per_config(all_results_df, SEARCH_PARAM_ORDER)

final_best = all_best_per_cfg.sort_values(SORT_BY, ascending=ASCENDING).iloc[0]

display_cols = SEARCH_PARAM_ORDER + [
    "stage",
    "epoch",
    "ece_small",
    "per_condition_calibration_error",
    "ece",
    "global_calibration_error",
    "test_loss",
]
print("Final best setting across both stages:")
display(final_best[display_cols].to_frame().T)

print("\nTop 20 configs (best epoch per config):")
display(all_best_per_cfg.loc[:, display_cols].sort_values(SORT_BY, ascending=ASCENDING).head(20).reset_index(drop=True))


In [ ]:
def write_df(df: pd.DataFrame, name: str) -> Path:
    path = run_output_dir / f"{name}_{run_stamp}.csv"
    df.to_csv(path, index=False)
    latest = run_output_dir / f"{name}_latest.csv"
    df.to_csv(latest, index=False)
    return path

files_written = {}
files_written["coarse_results"] = write_df(coarse_results_df, "coarse_results")
files_written["coarse_best_per_config"] = write_df(coarse_best_per_cfg, "coarse_best_per_config")
files_written["coarse_conditions"] = write_df(coarse_condition_df, "coarse_conditions")
files_written["narrow_results"] = write_df(narrow_results_df, "narrow_results")
files_written["narrow_best_per_config"] = write_df(narrow_best_per_cfg, "narrow_best_per_config")
files_written["narrow_conditions"] = write_df(narrow_condition_df, "narrow_conditions")
files_written["all_results"] = write_df(all_results_df, "all_results")
files_written["all_best_per_config"] = write_df(all_best_per_cfg, "all_best_per_config")
files_written["all_conditions"] = write_df(all_condition_df, "all_conditions")

summary = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "config_path": str(CONFIG_PATH),
    "loss_mode": LOSS_MODE,
    "loss_mode_canonical": LOSS_MODE_CANONICAL,
    "dataset": dataset_key,
    "output_dir": str(run_output_dir),
    "run_stamp": run_stamp,
    "fixed_model_params": FIXED_MODEL_PARAMS,
    "fixed_training_params": {
        "lr_category": None,
        "lr_category_decay": None,
    },
    "coarse_flags": {
        "points_per_param": COARSE_POINTS_PER_PARAM,
        "max_trials": COARSE_MAX_TRIALS,
        "use_full_grid": COARSE_USE_FULL_GRID,
        "epochs": COARSE_EPOCHS,
    },
    "narrow_flags": {
        "points_per_param": NARROW_POINTS_PER_PARAM,
        "max_trials": NARROW_MAX_TRIALS,
        "use_full_grid": NARROW_USE_FULL_GRID,
        "epochs": NARROW_EPOCHS,
    },
    "coarse_space": coarse_space,
    "narrow_space": narrow_space,
    "coarse_trial_count": len(coarse_grid),
    "narrow_trial_count": len(narrow_grid),
    "best": {
        "batch_size": int(final_best["batch_size"]),
        "lr_decay": float(final_best["lr_decay"]),
        "embed_dim": int(final_best["embed_dim"]),
        "amplification_rate": float(final_best["amplification_rate"]),
        "lr": float(final_best["lr"]),
        "best_stage": str(final_best["stage"]),
        "best_epoch": int(final_best["epoch"]),
        "ece_small": float(final_best["ece_small"]),
        "per_condition_calibration_error": float(final_best["per_condition_calibration_error"]),
        "ece": float(final_best["ece"]),
        "global_calibration_error": float(final_best["global_calibration_error"]),
        "test_loss": float(final_best["test_loss"]),
    },
    "files_written": {k: str(v) for k, v in files_written.items()},
}

summary_path = run_output_dir / f"run_summary_{run_stamp}.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
(run_output_dir / "run_summary_latest.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Files written:")
for k, v in files_written.items():
    print(f"- {k}: {v}")
print(f"- summary: {summary_path}")


In [ ]:
# Quick visual check of the objective trajectory by stage.
plot_df = all_results_df.sort_values(["stage", "epoch"]).copy()
fig, ax = plt.subplots(figsize=(8, 4))
for stage_name in ["coarse", "narrow"]:
    block = plot_df.loc[plot_df["stage"] == stage_name]
    if block.empty:
        continue
    best_by_epoch = block.groupby("epoch", as_index=False)["ece_small"].min()
    ax.plot(best_by_epoch["epoch"], best_by_epoch["ece_small"], marker="o", label=f"{stage_name} best ece_small")
ax.set_title("Best ece_small by epoch")
ax.set_xlabel("epoch")
ax.set_ylabel("ece_small")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()
